# STT_MultiLingva — Colab runner

Runs the multilingual transcriber on a free Colab T4 instead of local hardware.

## Read this before uploading anything

**Colab is Google infrastructure.** Audio uploaded here leaves your machine and
lands on Google's servers. If the recording is confidential, that is the same
disclosure you were avoiding by not using a transcription API — the model is
local, but the file is not.

Use this notebook for testing the pipeline on audio you are free to share. For a
confidential meeting, run the Docker image on a GPU inside your own perimeter;
`README.md` in this folder covers it.

**Runtime → Change runtime type → T4 GPU** before running the cells.


## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv


## 2. Install

`faster-whisper` pulls CTranslate2, which needs the cuDNN 9 runtime. Colab's
image ships CUDA but not always the matching cuDNN, so it is installed
explicitly rather than left to chance.

In [ ]:
!pip install -q faster-whisper==1.2.1
!pip install -q nvidia-cudnn-cu12==9.*
import os
# CTranslate2 loads cuDNN via the dynamic linker, which does not look inside
# pip packages on its own.
import nvidia.cudnn, pathlib
os.environ["LD_LIBRARY_PATH"] = str(pathlib.Path(nvidia.cudnn.__file__).parent / "lib") + ":" + os.environ.get("LD_LIBRARY_PATH", "")
print("cuDNN path added")


## 3. Fetch the transcriber

Cloned rather than pasted so the notebook and the script cannot drift apart.

In [ ]:
!git clone --depth 1 -b claude/current-directory-yxo7oy https://github.com/psalovsky/agentmemory /content/repo
%cd /content/repo/STT_MultiLingva
!ls


## 4. Provide audio

Either upload a file, or mount Drive and point at a path there. Both put the
audio on Google's servers — see the warning at the top.

In [ ]:
from google.colab import files
uploaded = files.upload()
AUDIO = next(iter(uploaded))
print("using", AUDIO)


### Alternative: Drive

Skip the upload cell above and run this one instead.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# AUDIO = '/content/drive/MyDrive/meeting.m4a'


## 5. Try five minutes first

A short excerpt shows whether language detection is behaving before you commit
the whole file to a session that Colab may reclaim mid-run.

In [ ]:
!ffmpeg -y -loglevel error -i "$AUDIO" -t 300 -ac 1 -ar 16000 /content/excerpt.wav
!python transcribe.py /content/excerpt.wav \
    --model large-v3 --device cuda --compute-type float16 \
    --languages ru,en --primary ru --out /content/out/excerpt


In [ ]:
print(open('/content/out/excerpt.txt').read()[:3000])


### Check the language split

The JSON carries per-line language and confidence. Lines sitting near the
threshold are where detection was shaky — if the wrong language dominates,
adjust `--threshold` or shorten `--window` before the full run.

In [ ]:
import json, collections
lines = json.load(open('/content/out/excerpt.json'))
print(collections.Counter(l['language'] for l in lines))
for l in lines:
    if l['language_probability'] < 0.75:
        print(f"{l['start']:7.1f}s  {l['language']}  p={l['language_probability']:.2f}  {l['text'][:70]}")


## 6. Full run

Colab disconnects idle sessions and caps runtime, so keep the tab open. On a T4
`large-v3` runs roughly 20-30x real time: a four-hour recording lands in about
ten minutes.

In [ ]:
import time
start = time.time()
!python transcribe.py "$AUDIO" \
    --model large-v3 --device cuda --compute-type float16 \
    --languages ru,en --primary ru --out /content/out/meeting
print(f"\n{(time.time() - start) / 60:.1f} minutes")


## 7. Download and clean up

Deleting the uploaded audio does not undo the upload -- Google still received
it. This only limits how long it sits in the session.

In [ ]:
from google.colab import files
for name in ['meeting.srt', 'meeting.txt', 'meeting.json']:
    files.download(f'/content/out/{name}')


In [ ]:
import os
os.remove(AUDIO)
os.path.exists('/content/excerpt.wav') and os.remove('/content/excerpt.wav')
print("session copies removed")


## Adding Armenian

Add `hy` to `--languages`. `large-v3` handles it at roughly 15.75% WER across
dialects out of the box. If Armenian is a large share of the recording, re-run
the windows the JSON marks `hy` through a fine-tuned model
(`Chillarmo/whisper-large-v3-turbo-armenian`) by passing it to `--model`.